# 02 — Baseline CNN
Time-boxed to 2-3 hours. Purpose: show transfer learning beats a from-scratch CNN. Do not tune further.

In [2]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

Working directory: C:\Users\Akshat Agarwal\Downloads\satellite-landuse-changedetection (1)\satellite-landuse-changedetection


In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
# Runs the standalone training script. Equivalent to:
#   python -m src.training.train_baseline
from src.training import train_baseline
train_baseline.main()


Using device: cuda


baseline epochs:   0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
import yaml, torch
from torch.utils.data import DataLoader, Subset
from src.data.datasets import EuroSATDataset
from src.data.transforms import get_eval_transform
from src.data.splits import spatial_block_split
from src.models.baseline_cnn import BaselineCNN
from src.training.engine import evaluate, load_checkpoint
from src.evaluation.metrics import per_class_f1, macro_f1, plot_confusion_matrix
from torch import nn

wwith open('config.yaml') as f:
    cfg = yaml.safe_load(f)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ds = EuroSATDataset(cfg['paths']['data_raw'], transform=get_eval_transform(cfg['data']['image_size']), download=False)
splits = spatial_block_split(ds, val_fraction=cfg['data']['val_fraction'], test_fraction=cfg['data']['test_fraction'], seed=cfg['seed'])
val_loader = DataLoader(Subset(ds, splits['val']), batch_size=cfg['data']['batch_size'])

model = BaselineCNN(num_classes=len(cfg['data']['eurosat_classes']), image_size=cfg['data']['image_size'])
model = load_checkpoint(model, f"{cfg['paths']['checkpoints']}/baseline_cnn.pt", device).to(device)

metrics = evaluate(model, val_loader, nn.CrossEntropyLoss(), device)
print('Baseline macro-F1:', metrics['macro_f1'])
print('Per-class F1:', per_class_f1(metrics['labels'], metrics['preds'], cfg['data']['eurosat_classes']))
plot_confusion_matrix(metrics['labels'], metrics['preds'], cfg['data']['eurosat_classes'],
                       save_path='../outputs/confusion_matrices/baseline_cnn_cm.png',
                       title='Baseline CNN Confusion Matrix')
